# Hugging Face 모델 탐색 실습

이번 실습에서는 Hugging Face에서 제공하는 다양한 인공지능 모델을 직접 찾아보고, Colab에서 실행해보겠습니다.

오늘은 모델을 직접 학습시키는 것이 아니라, 이미 학습되어 공개된 모델을 불러와 사용하는 방법을 연습합니다.

이번 시간에 실습할 task는 다음 4가지입니다.

1. 감정분석
2. 번역
3. 한국어 요약
4. 문장 유사도 비교


#라이브러리 설치

Hugging Face 모델을 사용하기 위해 필요한 라이브러리를 먼저 설치하겠습니다.

아래 코드는 실습에 필요한 기본 라이브러리를 설치하는 코드입니다.
처음 한 번만 실행하면 됩니다.

transformers: Hugging Face 모델을 불러오고 실행하는 라이브러리입니다.
sentencepiece: 일부 번역 및 요약 모델에서 사용하는 토크나이저 라이브러리입니다.
sentence-transformers: 문장을 벡터로 바꾸고 유사도를 계산할 때 사용하는 라이브러리입니다.

In [1]:
!pip install -q transformers sentencepiece sentence-transformers accelerate

In [2]:
import transformers
import sentence_transformers

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

transformers: 5.12.1
sentence-transformers: 5.6.0


#Hugging Face 모델 검색 방법

Hugging Face에는 다양한 인공지능 모델이 공개되어 있습니다.

이번 실습에서는 원하는 task에 맞는 모델을 직접 검색해보고, Colab에서 실행해보겠습니다.

모델을 찾는 순서는 다음과 같습니다.

1. Hugging Face 사이트에 접속합니다.
2. 상단 메뉴에서 `Models`를 클릭합니다.
3. 왼쪽 메뉴에서 원하는 `Task`를 선택합니다.
4. 검색창에 원하는 키워드를 입력합니다.
5. 마음에 드는 모델 페이지에 들어갑니다.
6. 모델 이름을 복사합니다.
7. Colab 코드의 `model_name` 부분에 붙여넣습니다.

모델 이름은 보통 다음과 같은 형태입니다.

```text
사용자명/모델명
```

예를 들어 다음과 같은 이름을 사용할 수 있습니다.

```text
distilbert-base-uncased-finetuned-sst-2-english
eenzeenee/t5-base-korean-summarization
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

처음에는 어떤 모델을 골라야 할지 헷갈릴 수 있습니다.
그럴 때는 다운로드 수가 많거나, 모델 카드에 사용 예시가 잘 적혀 있는 모델을 먼저 선택해보면 좋습니다.


#1. 감정분석
- sentiment-analysis
- text-classificatio

In [29]:
from transformers import pipeline

classifier = pipeline("text-classification", "tabularisai/multilingual-sentiment-analysis")

sentences = [
    "오늘 날씨도 좋고 기분이 최고야!",
    "그냥 평범한 하루였어.",
    "배송도 느리고 제품도 별로네요.",
    "ㅋㅋ;;"
]

# 결과 출력
results = classifier(sentences)
for sentence, result in zip(sentences, results):
    print(f"문장: {sentence} -> 결과: {result['label']} (확률: {result['score']:.4f})")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

문장: 오늘 날씨도 좋고 기분이 최고야! -> 결과: Very Positive (확률: 0.5559)
문장: 그냥 평범한 하루였어. -> 결과: Neutral (확률: 0.6770)
문장: 배송도 느리고 제품도 별로네요. -> 결과: Negative (확률: 0.8089)
문장: ㅋㅋ;; -> 결과: Neutral (확률: 0.4159)


#2. 번역
- translation

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

ko_text = "허깅페이스를 활용한 자연어 처리 실습은 매우 흥미롭습니다."


inputs = tokenizer(ko_text, return_tensors="pt", padding=True)
outputs = model.generate(**inputs)


en_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("원문:", ko_text)
print("번역:", en_text)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

원문: 허깅페이스를 활용한 자연어 처리 실습은 매우 흥미롭습니다.
번역: And the natural language processing that we've done with the Huggingspace is very interesting.


#3. 요약
- summarization
- text2text-generation

In [21]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 1. 외계어 버그를 방지하기 위해 파이프라인 대신 전용 클래스로 정석 호출합니다.
model_name = "EbanLee/kobart-summary-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2. 요약할 한국어 본문
long_text = """
인공지능(AI) 기술이 급격히 발전하면서 전 세계 기술 기업들이 오픈소스 모델 공유 플랫폼인 허깅페이스로 모여들고 있습니다.
과거에는 각 기업과 연구소들이 자신들이 개발한 고성능 모델의 가중치(Weights)를 철저히 비밀로 유지하는 경향이 강했습니다.
하지만 최근에는 이미 학습된 모델을 오픈소스로 공개하고 전 세계 개발자들이 이를 자유롭게 변형하여 사용하는 생태계가 구축되었습니다.
전문가들은 이러한 개방형 공유 생태계가 막대한 컴퓨팅 자원과 개발 시간을 절약해 주어, 전 세계적인 AI 기술 상용화 속도를 가속화하고 있다고 분석합니다.
"""

# 3. 토크나이징 연산 진행
inputs = tokenizer(long_text, return_tensors="pt", max_length=512, truncation=True)

# 4. 요약 문장 생성 (안정적인 요약을 위해 do_sample=False 설정)
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=64,
    min_length=10,
    num_beams=4,
    do_sample=False
)

# 5. 글자 깨짐 방지를 위해 완벽하게 디코딩 처리
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("=== [KoBART 한국어 요약 결과] ===")
print(summary_text)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

=== [KoBART 한국어 요약 결과] ===
전문가들은 개방형 공유 생태계가 막대한 컴퓨팅 자원과 개발 시간을 절약해 전 세계적인 AI 기술 상용화 속도를 가속화하고 있다고 분석한다.


#4. 문장 유사도 비교
- sentence-similarity
- sentence-transformers 라이브러리 사용

In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer


model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

base_sentence = "오늘 날씨가 좋아서 밖에 나가고 싶다."

compare_sentences = [
    "날씨가 화창해서 야외 활동을 하기에 딱 좋은 날이네.",  # 의미가 매우 유사한 문장
    "내일은 코딩 실습 시험이 있으니 공부를 해야 한다.",    # 의미가 전혀 다른 문장
    "비가 오고 흐려서 하루 종일 집에만 있고 싶다."         # 반대 느낌이지만 '날씨와 외출'이라는 주제가 겹치는 문장
]

base_embedding = model.encode(base_sentence)
compare_embeddings = model.encode(compare_sentences)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f" 기준 문장: '{base_sentence}'\n")
print("--- 다른 문장들과의 의미 유사도 결과 ---")

for sentence, embedding in zip(compare_sentences, compare_embeddings):
    similarity = cosine_similarity(base_embedding, embedding)
    print(f"비교 문장: '{sentence}'")
    print(f"👉 유사도 점수: {similarity:.4f} ({similarity*100:.1f}% 일치)\n")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 기준 문장: '오늘 날씨가 좋아서 밖에 나가고 싶다.'

--- 다른 문장들과의 의미 유사도 결과 ---
비교 문장: '날씨가 화창해서 야외 활동을 하기에 딱 좋은 날이네.'
👉 유사도 점수: 0.8294 (82.9% 일치)

비교 문장: '내일은 코딩 실습 시험이 있으니 공부를 해야 한다.'
👉 유사도 점수: 0.2192 (21.9% 일치)

비교 문장: '비가 오고 흐려서 하루 종일 집에만 있고 싶다.'
👉 유사도 점수: 0.5825 (58.2% 일치)

